In [2]:
import json
import math
import torch
import pandas as pd

from pathlib import Path
from tqdm.auto import tqdm

from summac.model_summac import SummaCZS
from rouge_score.rouge_scorer import RougeScorer

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

def require(condition, message):
    if not condition:
        raise ValueError(message)

CONFIG = {
    "summac": {
        "model_name": "vitc",
        "granularity": "sentence",
        "op1": "max",
        "op2": "mean",
        "use_ent": True,
        "use_con": False,
        "max_doc_sents": 100,
    },
    "rouge": {
        "metric": "rougeL",
        "use_stemmer": True,
        "references": "max",
    },
    "batch_size": 8,
}

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

Device: cuda:0
PyTorch: 2.15.0+cu132
CUDA: True


In [3]:
TEST_PATH = ROOT / "data" / "processed" / "test.jsonl"
OUT_DIR = ROOT / "outputs" / "test"
INPUT_CSV = OUT_DIR / "overall_alignscore_results.csv"
OUTPUT_CSV = OUT_DIR / "summac_rouge_eval.csv"

assert TEST_PATH.exists(), f"Missing test dataset: {TEST_PATH}"
assert INPUT_CSV.exists(), f"Missing AlignScore output: {INPUT_CSV}"


print("Input:", INPUT_CSV)
print("Output:", OUTPUT_CSV)

Input: c:\Users\Ramski\Documents\VSCode\Projects\csc173-midterm\outputs\test\overall_alignscore_results.csv
Output: c:\Users\Ramski\Documents\VSCode\Projects\csc173-midterm\outputs\test\summac_rouge_eval.csv


In [4]:
with open(TEST_PATH, "r", encoding="utf-8") as f:
    data = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

candidate_df = pd.read_csv(INPUT_CSV)

candidates = {
    str(row["id"]): row
    for _, row in candidate_df.iterrows()
}


assert len(candidates) == len(data), "Article count mismatch."

print("Test articles:", len(data))
print("Selected summaries:", len(candidates))


Test articles: 615
Selected summaries: 615


In [5]:
scorer = SummaCZS(
    **CONFIG["summac"],
    device=DEVICE,
    use_cache=False,
    imager_load_cache=False,
)

rouge = RougeScorer(
    [CONFIG["rouge"]["metric"]],
    use_stemmer=CONFIG["rouge"]["use_stemmer"],
)

print("SummaCZS loaded.")
print("ROUGE-L loaded.")

SummaCZS loaded.
ROUGE-L loaded.


In [9]:
if OUTPUT_CSV.exists():
    rows = pd.read_csv(OUTPUT_CSV).to_dict("records")
    completed_ids = {str(r["id"]) for r in rows}
else:
    rows = []
    completed_ids = set()

for article in tqdm(data, desc="SummaC / ROUGE-L"):

    article_id = str(article["id"])

    if article_id in completed_ids:
        continue

    candidate_summary = str(
        candidates[article_id]["selected_summary"]
        ).strip()
    
    baseline_summary = str(
        candidates[article_id]["baseline_summary"] 
        ).strip()

    def require(condition, message):
        if not condition:
            raise ValueError(message)

    require(
        bool(scorer.imager.split_sentences(candidate_summary)),
        "Empty summary sentences",
    )

    with torch.inference_mode():
        summac_candidates_result = scorer.score(
            [article["abstract"]],
            [candidate_summary],
            batch_size=CONFIG["batch_size"],
        )

    summac_candidate_score = float(summac_candidates_result["scores"][0])

    with torch.inference_mode():
            summac_baseline_result = scorer.score(
                [article["abstract"]],
                [baseline_summary],
                batch_size=CONFIG["batch_size"],
            )
    
    summac_baseline_score = float(summac_baseline_result["scores"][0])

    if not math.isfinite(summac_candidate_score):
        raise ValueError(
            f"Invalid SummaC score: {article_id}"
        )
    
    if not math.isfinite(summac_baseline_score):
        raise ValueError(
            f"Invalid SummaC score: {article_id}"
        )

    candidate_rouge_score = max(
        rouge.score(reference, candidate_summary)[CONFIG["rouge"]["metric"]].fmeasure
        for reference in article["human_tldrs"]
    )

    if not math.isfinite(candidate_rouge_score):
        raise ValueError(
            f"Invalid ROUGE-L score: {article_id}"
        )

    baseline_rouge_score = max(
        rouge.score(reference, baseline_summary)[CONFIG["rouge"]["metric"]].fmeasure
        for reference in article["human_tldrs"]
    )

    if not math.isfinite(baseline_rouge_score):
        raise ValueError(
            f"Invalid ROUGE-L score: {article_id}"
        )

    rows.append({
        "id": article_id,
        "baseline_summary": baseline_summary,
        "candidate_summary": candidate_summary,
        "baseline_summac": summac_baseline_score,
        "baseline_rougeL": baseline_rouge_score,
        "candidate_summac": summac_candidate_score,
        "candidate_rougeL": candidate_rouge_score,
    })

    completed_ids.add(article_id)

    # Save after every article
    pd.DataFrame(rows).to_csv(
        OUTPUT_CSV,
        index=False,
        encoding="utf-8-sig"
    )

    # Clear SummaC cache
    scorer.imager.cache.clear()

print("Evaluation complete!")
print("Articles evaluated:", len(rows))
print("Output:", OUTPUT_CSV)

SummaC / ROUGE-L:   0%|          | 0/615 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/235M [00:00<?, ?B/s]

Evaluation complete!
Articles evaluated: 615
Output: c:\Users\Ramski\Documents\VSCode\Projects\csc173-midterm\outputs\test\summac_rouge_eval.csv


In [10]:
results_df = pd.DataFrame(rows)

print("Evaluation results")
print("------------------")

print("Articles:", len(results_df))
print("Mean Candidate SummaC:", results_df["candidate_summac"].mean())
print("Mean Candidate ROUGE-L:", results_df["candidate_rougeL"].mean())
print("Mean Baseline SummaC:", results_df["baseline_summac"].mean())
print("Mean Baseline ROUGE-L:", results_df["baseline_rougeL"].mean())

print("\nDescriptive statistics:")
print(
    results_df[["candidate_summac", "candidate_rougeL"]].describe()
)
print(
    results_df[["baseline_summac", "baseline_rougeL"]].describe()
)

Evaluation results
------------------
Articles: 615
Mean Candidate SummaC: 0.8753065708933807
Mean Candidate ROUGE-L: 0.36584155511093186
Mean Baseline SummaC: 0.761174256480261
Mean Baseline ROUGE-L: 0.36751780812135637

Descriptive statistics:
       candidate_summac  candidate_rougeL
count        615.000000        615.000000
mean           0.875307          0.365842
std            0.279645          0.185087
min            0.015757          0.062500
25%            0.983430          0.243163
50%            0.990957          0.315789
75%            0.993160          0.421053
max            0.996741          1.000000
       baseline_summac  baseline_rougeL
count       615.000000       615.000000
mean          0.761174         0.367518
std           0.354050         0.187188
min           0.008886         0.086957
25%           0.534101         0.244898
50%           0.989589         0.315789
75%           0.992850         0.420204
max           0.995997         1.000000


In [12]:
summary_stats = {
    "total_articles": len(results_df),
    "mean_baseline_summac": float(results_df["baseline_summac"].mean()),
    "mean_baseline_rougeL": float(results_df["baseline_rougeL"].mean()),
    "std_baseline_summac": float(results_df["baseline_summac"].std()),
    "std_baseline_rougeL": float(results_df["baseline_rougeL"].std()),
    "mean_candidate_summac": float(results_df["candidate_summac"].mean()),
    "mean_candidate_rougeL": float(results_df["candidate_rougeL"].mean()),
    "std_candidate_summac": float(results_df["candidate_summac"].std()),
    "std_candidate_rougeL": float(results_df["candidate_rougeL"].std()),
    
}

CANDIDATE_STATS_PATH = OUT_DIR / "candidate_summac_rouge_summary.json"


with open(CANDIDATE_STATS_PATH, "w", encoding="utf-8") as f:
    json.dump(summary_stats, f, indent=4)

print("Statistics saved:", CANDIDATE_STATS_PATH)

Statistics saved: c:\Users\Ramski\Documents\VSCode\Projects\csc173-midterm\outputs\test\candidate_summac_rouge_summary.json
